# Chapter 4: How Information Flows — Polar Decomposition

**What you'll learn:** How each layer transition decomposes into rotation (direction change) and stretching (magnitude change), and what each component means computationally.

**Key concept:** T = U·P where U is rotation and P is stretching. This is the heart of the framework.

**Time:** ~20 minutes

## 4.1 Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import layer_time_geometry as core
import ltg

model = ltg.load_model("Qwen/Qwen2.5-7B", device="cuda")

## 4.2 The Transition Operator

Between layer l and l+1, the hidden states change. We approximate this as a linear map:

$$\tilde{H}^{(l+1)} \approx \tilde{H}^{(l)} \cdot T^{(l)}$$

where $T^{(l)}$ is found by least squares (just linear regression on matrices).

In [ ]:
result = ltg.analyse("If x equals 3 and y equals 2x plus 1 then what is y", model=model)

# The polar decomposition is already computed
print(f"Number of layer transitions: {len(result.polar_U)}")
print(f"Each U matrix shape: {result.polar_U[0].shape}")
print(f"Each P matrix shape: {result.polar_P[0].shape}")

## 4.3 Understanding Rotation (U) vs Stretching (P)

**Rotation U:** Changes directions without changing magnitudes. Think of it as the model *redirecting attention* — same amount of information, different configuration.

**Stretching P:** Amplifies some directions and compresses others. Think of it as the model *making decisions* — boosting important features, suppressing irrelevant ones.

Let's visualise how much each layer rotates vs stretches.

In [ ]:
# Compute deviation from identity for both U and P
rotation_magnitude = []
stretch_magnitude = []

I_k = np.eye(result.k)
for l in range(len(result.polar_U)):
    U_dev = np.linalg.norm(result.polar_U[l] - I_k, 'fro')
    P_dev = np.linalg.norm(result.polar_P[l] - I_k, 'fro')
    rotation_magnitude.append(U_dev)
    stretch_magnitude.append(P_dev)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

layers = range(len(rotation_magnitude))

# Side-by-side profiles
ax1.plot(layers, rotation_magnitude, 'o-', color='#4477AA', label='Rotation ||U - I||', markersize=3)
ax1.plot(layers, stretch_magnitude, 's-', color='#EE6677', label='Stretch ||P - I||', markersize=3)
ax1.set_xlabel('Layer')
ax1.set_ylabel('Deviation from identity')
ax1.set_title('Rotation vs Stretching Magnitude per Layer')
ax1.legend()

# Ratio: which dominates?
ratio = np.array(stretch_magnitude) / (np.array(rotation_magnitude) + 1e-8)
ax2.bar(layers, ratio, color=['#EE6677' if r > 1 else '#4477AA' for r in ratio], alpha=0.7)
ax2.axhline(1.0, color='grey', linestyle='--', label='Equal')
ax2.set_xlabel('Layer')
ax2.set_ylabel('Stretch / Rotation ratio')
ax2.set_title('Which Dominates? (red = stretch, blue = rotation)')
ax2.legend()

plt.tight_layout()
plt.show()

## 4.4 The Condition Number — How Selective Is Each Layer?

The eigenvalues of P tell us how much each direction is amplified or compressed.
- **Condition number κ = σ_max / σ_min** — high κ means the layer is very selective.
- **Effective rank** — how many directions are actively used.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(result.condition_numbers, 'o-', color='#AA3377', markersize=4, linewidth=2)
ax1.set_xlabel('Layer', fontsize=12)
ax1.set_ylabel('Condition number κ', fontsize=12)
ax1.set_title('Condition Number — Layer Selectivity', fontsize=14)
peak = result.condition_numbers.argmax()
ax1.annotate(f'Peak: layer {peak}\nκ = {result.condition_numbers[peak]:.1f}',
             xy=(peak, result.condition_numbers[peak]),
             xytext=(peak-5, result.condition_numbers.max()*0.8),
             fontsize=10, arrowprops=dict(arrowstyle='->'))

ax2.plot(result.effective_ranks, 'o-', color='#66CCEE', markersize=4, linewidth=2)
ax2.set_xlabel('Layer', fontsize=12)
ax2.set_ylabel('Effective rank', fontsize=12)
ax2.set_title('Effective Rank — Active Dimensions', fontsize=14)

plt.tight_layout()
plt.show()

print(f"Peak condition number: κ = {result.condition_numbers.max():.1f} at layer {peak}")
print(f"Interpretation: layer {peak} amplifies its top direction {result.condition_numbers.max():.0f}x more than its weakest.")

## 4.5 Inspecting the Eigenvalue Spectrum of a Single Layer

Let's zoom into the layer with the highest condition number and see its eigenvalue distribution.

In [ ]:
peak_layer = result.condition_numbers.argmax()
P = result.polar_P[peak_layer]
sigmas = np.linalg.svd(P, compute_uv=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Full spectrum
ax1.semilogy(sigmas, 'o-', markersize=3, color='#228833')
ax1.set_xlabel('Index')
ax1.set_ylabel('Eigenvalue (log scale)')
ax1.set_title(f'Eigenvalue Spectrum of P at Layer {peak_layer}')
ax1.axhline(1.0, color='grey', linestyle='--', label='σ = 1 (no change)')
ax1.legend()

# Top 20 eigenvalues
ax2.bar(range(20), sigmas[:20], color='#228833', alpha=0.7)
ax2.axhline(1.0, color='grey', linestyle='--')
ax2.set_xlabel('Index')
ax2.set_ylabel('Eigenvalue')
ax2.set_title(f'Top 20 Eigenvalues at Layer {peak_layer}')

plt.tight_layout()
plt.show()

print(f"σ_max = {sigmas[0]:.3f} (most amplified direction)")
print(f"σ_min = {sigmas[-1]:.6f} (most compressed direction)")
print(f"Directions with σ > 1 (amplified): {(sigmas > 1).sum()}")
print(f"Directions with σ < 1 (compressed): {(sigmas < 1).sum()}")

## 4.6 Comparing Tasks

Does the polar decomposition differ by task type?

In [ ]:
prompts = {
    'Arithmetic': 'What is 144 divided by 12?',
    'Reasoning': 'If it rains then the ground is wet. The ground is dry. Did it rain?',
    'Retrieval': 'What is the boiling point of water?',
}

fig, ax = plt.subplots(figsize=(10, 5))

for name, text in prompts.items():
    r = ltg.analyse(text, model=model, compute_dependency=False)
    ax.plot(r.condition_numbers, linewidth=2, label=f'{name} (peak κ={r.condition_numbers.max():.0f})')

ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Condition number κ', fontsize=12)
ax.set_title('Condition Number Profiles by Task Type', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

## 4.7 Key Takeaways

1. Every layer transition = **rotation + stretching** (polar decomposition).
2. **Rotation** redistributes information; **stretching** amplifies/compresses.
3. **Condition number** measures layer selectivity — high κ = sharp decisions.
4. **Effective rank** measures how many directions the layer uses.
5. Late layers tend to have higher condition numbers (more selective).

## Exercise

Compare the rotation vs stretch profiles for the same prompt on two different models (e.g., Qwen2.5-7B vs Qwen2.5-1.5B). Do larger models rotate more or stretch more?

In [ ]:
# Your turn!
# model_small = ltg.load_model("Qwen/Qwen2.5-1.5B", device="cuda")
# r_small = ltg.analyse("What is 2 + 2?", model=model_small)
# r_large = ltg.analyse("What is 2 + 2?", model=model)
# Compare condition_numbers, effective_ranks